In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")

In [3]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage('테슬라는 한달 전에 비해 주가가 올랐나 내렸나?')])

AIMessage(content='실시간 주가 정보는 계속 변동하기 때문에 제가 지금 바로 "올랐다" 또는 "내렸다"고 단정적으로 말씀드리기는 어렵습니다. 제 정보는 특정 시점까지의 데이터에 기반하며, 현재 시장 상황을 실시간으로 반영하지는 않습니다.\n\n하지만 직접 확인하실 수 있는 방법은 다음과 같습니다:\n\n1.  **금융 웹사이트/앱 접속:** 네이버 금융, 카카오 증권, 구글 금융, 야후 파이낸스 등의 금융 웹사이트나 앱에 접속하세요.\n2.  **테슬라 (TSLA) 검색:** 검색창에 "테슬라" 또는 "TSLA"를 입력하여 종목을 검색합니다.\n3.  **1개월 차트 확인:** 대부분의 사이트에서 주가 차트와 함께 1개월(1M) 또는 30일(30D) 버튼을 제공합니다. 이 버튼을 클릭하여 한 달간의 주가 추이를 확인하시면 됩니다.\n\n현재 시점에서 직접 확인하시는 것이 가장 정확합니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d5caf-7789-7142-b8e7-04465a7cbdca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 744, 'total_tokens': 761, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 523}})

In [5]:
import yfinance as yf

tsla = yf.Ticker("TSLA")


In [8]:
hist = tsla.history(period="1mo")

In [9]:
type(hist)

pandas.DataFrame

In [10]:
hist

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-03-03 00:00:00-05:00,395.089996,396.339996,385.390015,392.429993,62617300,0.0,0.0
2026-03-04 00:00:00-05:00,397.850006,408.329987,394.579987,405.940002,68305500,0.0,0.0
2026-03-05 00:00:00-05:00,401.570007,408.619995,399.420013,405.549988,51925900,0.0,0.0
2026-03-06 00:00:00-05:00,398.089996,402.350006,394.209991,396.730011,64054600,0.0,0.0
2026-03-09 00:00:00-04:00,390.049988,401.589996,381.399994,398.679993,67018900,0.0,0.0
2026-03-10 00:00:00-04:00,402.220001,406.589996,398.190002,399.239990,59258700,0.0,0.0
2026-03-11 00:00:00-04:00,402.279999,416.380005,402.149994,407.820007,62559900,0.0,0.0
2026-03-12 00:00:00-04:00,405.179993,406.500000,394.649994,395.010010,60973800,0.0,0.0
2026-03-13 00:00:00-04:00,399.170013,400.200012,389.950012,391.200012,58504100,0.0,0.0


In [12]:
hist.to_markdown()

'| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-03-03 00:00:00-05:00 | 395.09 | 396.34 | 385.39 |  392.43 | 6.26173e+07 |           0 |              0 |\n| 2026-03-04 00:00:00-05:00 | 397.85 | 408.33 | 394.58 |  405.94 | 6.83055e+07 |           0 |              0 |\n| 2026-03-05 00:00:00-05:00 | 401.57 | 408.62 | 399.42 |  405.55 | 5.19259e+07 |           0 |              0 |\n| 2026-03-06 00:00:00-05:00 | 398.09 | 402.35 | 394.21 |  396.73 | 6.40546e+07 |           0 |              0 |\n| 2026-03-09 00:00:00-04:00 | 390.05 | 401.59 | 381.4  |  398.68 | 6.70189e+07 |           0 |              0 |\n| 2026-03-10 00:00:00-04:00 | 402.22 | 406.59 | 398.19 |  399.24 | 5.92587e+07 |           0 |              0 |\n| 2026-03-11 00:00:00-04:00 | 402.28 | 416.38 | 402.15 |  407.82 | 6.25599e+07 |           0 | 

In [14]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: TSLA)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d,5d,1mo,3mo,6mo,1y,2y,5y,10y,ytd,max)")

In [15]:
from langchain.tools import tool

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    return history.to_markdown()

In [16]:
from langchain.agents import create_agent

agent = create_agent(model="google_genai:gemini-2.5-flash", tools=[get_yf_stock_history])

In [17]:
from langchain.messages import HumanMessage

agent.invoke({
    "messages": [HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐나 내렸나?')]
})

{'messages': [HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐나 내렸나?', additional_kwargs={}, response_metadata={}, id='b1658acd-11c0-4f3c-b83a-151a0c731d25'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"period": "1mo", "ticker": "TSLA"}}'}, '__gemini_function_call_thought_signatures__': {'e098f195-e1d9-4a3e-a6a4-035a72b670ab': 'CuEWAb4+9vu7ynNNTZ6doH5stfjeHUcStVyFiu+hwNSgNzludgyA6dHVMARWKu0IeclJWsXtM6m2H/hV6XXKLBT6Ypb75lKFY/9QIl5LhLiQKa9BHgh0UcCPLAtS2V0HG05vAvU+2qBgC1eZHMBqQRt78t3qbl2vVBu0cdigsYRBtzK2T0WNDkDrLaE38nS+qTAaYoGHQ9bnVw6rlFTDd5TeyEf58fSVbfYiR0Ubfr/U+3n+JIN5zWdXZfJVZZth9CtcfSOnVwniXBakT+WD4DXIppxBsJ8CI3rN1qUaZeCtM6gnNAwT3p2d4lv6K8/fBkwc1KLfuCCNq5QiElGyYJIuNlhBWehQ2JTkK+5md7PygBEPLqXrN030ChBQqy0/F7a6jEF7fY8m7iXDebwZMvlKUq+5mW/VMLt1beU6ZCdHZK5L8PT5Xv8ee78Wx1ohBwzdB8IpVtrT8JJpH6TZ1gqgTPT1DQnT/CjZO/LQknM92C5GEMPLXpLj66TxaGBU5zXoueV9MDJYAe4OUzcfLzVet0EpmkUj6w4nZWH4RX5G8tUHAqY+ZIsxJzN05jHOBz36wuXsjyQ

In [18]:
from langchain.messages import HumanMessage

agent.invoke({
    "messages": [HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐는지 내렸는지 분석해줘')]
})

{'messages': [HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐는지 내렸는지 분석해줘', additional_kwargs={}, response_metadata={}, id='21b17bf8-2ab9-4d5a-8b81-143a6a062417'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"period": "1mo", "ticker": "TSLA"}}'}, '__gemini_function_call_thought_signatures__': {'79330167-175d-4859-b6b6-73dec598f308': 'CocHAb4+9vuJq/jyC+UZGJyQZZbO5tEKrifgLRyVZZT0LMw9xe84Jwmkce1iEi2IPevHWiAgVSPd02dzn+h7b2/yo0Qvgz/WLO+ioU72s573XYha7no+L+0VRWmLXskMGCJb4rK6V51pGjXcPsSOl99darg5qlOX2XWEuOzG3rK8shc1sAYitGDFz0U+/J1U2Xdxz7lmGe3d1yB1BZHN7CRPbQLdke+bayqcSQc3gisDSbQFyCN+k7mvpmL+KIPPFT45xvMb9xnIADKDtq+VrzC7NsomeTHtTDRIFSv4XVG4opTpQnYN6LV03J2yOfscrts7O2kp2DiRkG1N9sLuFwNP/XovcYMrx4Sf264sZRVfp+YjeP5quYa0I0VzG7TS95lXgTFPn+CqK/q5CUfQBm7cjmdK+sZGwr9loS2tCTBjRTC64TFT2QK78jEVGP+bgTavGNs24cCBYeBL5vy2c6l/EkyAsqRq4fMs0H02B+PujrZIWdhHUXptfo0Wu0Oktv+GgkOwzIOMkj6k7dM9t6pEs3ZaNtgkYUx5SKzPBiiGhGUrm9ZBd+akQ2gMuG8lZJEQx